# Blood Work Analysis Pipeline
A two-stage LLM pipeline:
- **Stage 1:** Extract and flag abnormal values from a blood report
- **Stage 2:** Generate a health summary and Indian diet plan based on flagged values

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(model="qwen/qwen3.8-27b")

## Sample Blood Report

In [2]:
with open("blood_work.txt", "r") as f:
    blood_report = f.read()

print(blood_report[:200])

Patient: Rajesh Sharma, Age 48, Male
Date: May 7, 2026

COMPLETE BLOOD COUNT (CBC)
--------------------------
Hemoglobin:        15.1 g/dL        (Normal: 13.5â€“17.5)
Hematocrit:        44%          


## Stage 1 — Extract and Flag Abnormal Values

In [3]:
extraction_prompt = f"""
You are a medical data extraction assistant.

From the blood report below, extract ALL test values and classify each one as HIGH, LOW, or NORMAL 
based on the reference ranges provided in the report.

Format your response as:
- Test Name: value | Status: HIGH/LOW/NORMAL | Reference: range

Blood Report:
{blood_report}
"""

extraction_response = llm.invoke(extraction_prompt)
extracted_values = extraction_response.text

print("=== STAGE 1: EXTRACTED VALUES ===")
print(extracted_values)

=== STAGE 1: EXTRACTED VALUES ===
Here are the extracted blood test values classified based on the provided reference ranges:

**COMPLETE BLOOD COUNT (CBC)**
- Hemoglobin: 15.1 g/dL | Status: NORMAL | Reference: 13.5–17.5
- Hematocrit: 44% | Status: NORMAL | Reference: 41–53%
- WBC: 6.8 x10^3/uL | Status: NORMAL | Reference: 4.5–11.0
- Platelets: 220 x10^3/uL | Status: NORMAL | Reference: 150–400

**LIPID PANEL**
- Total Cholesterol: 238 mg/dL | Status: HIGH | Reference: <200
- LDL Cholesterol: 162 mg/dL | Status: HIGH | Reference: <100
- HDL Cholesterol: 36 mg/dL | Status: LOW | Reference: >40
- Triglycerides: 188 mg/dL | Status: HIGH | Reference: <150

**METABOLIC PANEL**
- Glucose (Fasting): 92 mg/dL | Status: NORMAL | Reference: 70–99
- HbA1c: 5.3% | Status: NORMAL | Reference: <5.7%
- Creatinine: 1.0 mg/dL | Status: NORMAL | Reference: 0.7–1.3
- eGFR: 82 mL/min | Status: NORMAL | Reference: >60

**LIVER FUNCTION**
- ALT: 28 U/L | Status: NORMAL | Reference: 7–40
- AST: 25 U/L | St

## Stage 2 — Health Summary and Indian Diet Plan

In [4]:
diet_prompt = f"""
You are a clinical nutritionist specializing in Indian dietary habits.

Based on the blood work analysis below, write:
1. A short health summary in 4-5 lines explaining the patient's condition in simple language
2. A short, practical Indian diet plan having only two sections (1) Foods to avoid (2) Foods to eat more of. 
   Do not include any other sections in diet plan.

Blood Work Analysis:
{extracted_values}
"""

diet_response = llm.invoke(diet_prompt)

print("=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===")
print(diet_response.text)

=== STAGE 2: HEALTH SUMMARY & DIET PLAN ===
**Health Summary**
Your blood tests show normal blood counts, liver function, and sugar levels, which is good. However, you have high bad cholesterol (LDL) and triglycerides, along with low good cholesterol (HDL). This imbalance increases the risk of heart disease and clogged arteries. You do not have diabetes or kidney issues, but your lipid profile needs attention. Small dietary changes can help lower these cholesterol levels and protect your heart health.

**Indian Diet Plan**

**Foods to Avoid**
*   **Deep-fried snacks and sweets:** Avoid samosas, pakoras, puris, and fried namkeen. Stop consuming sugary sweets like gulab jamun, barfi, or jelly.
*   **Refined carbohydrates and white flour:** Reduce intake of maida-based foods such as naan, pav, white bread, and biscuits.
*   **Full-fat dairy and red meat:** Limit full-cream milk, curd (dahi), paneer, ghee, and red meats like mutton or chicken thighs.
*   **Processed and salty foods:** Cut 